In [1]:
import os
import json
import yaml
from typing import Iterable, List, Union

Find checkpoints with the given conditions

In [2]:
def matches_conditions(config_data, conditions):
    """Recursively check if config_data satisfies the given conditions."""
    for key, expected_value in conditions.items():
        if key not in config_data:
            return False

        actual_value = config_data[key]
        if isinstance(expected_value, dict):
            if not isinstance(actual_value, dict):
                return False
            if not matches_conditions(actual_value, expected_value):
                return False
        elif actual_value != expected_value:
            return False
    return True


def load_config_file(config_path: str):
    """Load configuration based on file extension. Supports .json / .yaml / .yml."""
    extension = os.path.splitext(config_path)[1].lower()
    with open(config_path, "r", encoding="utf-8") as config_file:
        if extension == ".json":
            return json.load(config_file)
        if extension in (".yaml", ".yml"):
            return yaml.safe_load(config_file)
        raise ValueError(f"Unsupported configuration format: {extension}")


def find_configs_by_conditions(
    conditions: dict,
    root_dir: str,
    filenames: Union[str, Iterable[str]] = ("config.yaml", "config.json"),
) -> List[str]:
    """Recursively find directories whose config file matches the given conditions."""
    if isinstance(filenames, str):
        target_filenames = {filenames}
    else:
        target_filenames = set(filenames)

    matched_directories = []

    for current_dir, _, files in os.walk(root_dir):
        matching_filenames = target_filenames.intersection(files)
        for filename in matching_filenames:
            config_path = os.path.join(current_dir, filename)
            try:
                config_data = load_config_file(config_path)
                if isinstance(config_data, dict) and matches_conditions(config_data, conditions):
                    matched_directories.append(os.path.abspath(current_dir))
                    break
            except Exception as exc:
                print(f"[Error] Failed to read {config_path}: {exc}")

    return matched_directories


In [5]:
checkpoints_root_dir = "./checkpoints"

condition_clf_mixer_qtm_salton_sea = {
    # "load_specific_parts": ["encoder"],
    "model": "clf_mixer_attnpl_t",
    "mixer_model_config": {
        # "d_model": 64,
        # "attn_layer_idx": [],
        # "ssm_cfg": {"layer": "Mamba2"}
    },
    "dataset": "QTMSaltonSea",
    # "Mf": 4.5,
    # "Twindow": 180,
    # "Tfore": 20
}

condition_etas_pnr_1z = {
    "model": "etas",
    # "mixer_model_config": {
    #     # "d_model": 64,
    #     # "attn_layer_idx": [0],
    #     # "ssm_cfg": {"layer": "Mamba2"}
    # },
    # "predict_b": False,
    # "features_input_keys": ["mag"],  # "log_inter_times"
    "bg_model": "mamba",
    "dataset": "PNR_1z",
}

condition_rtpp_chuandian = {
    "model": "rtpp",
    # "mixer_model_config": {
    #     # "d_model": 64,
    #     # "attn_layer_idx": [],
    #     # "ssm_cfg": {"layer": "Mamba2"}
    # },
    # "dMag": 0.1,
    "dataset": "ChuanDian",
}

condition_lstm_chuandian = {
    "model": "lstm",
    "dataset": "ChuanDian",
}

condition_reg_mixer_chuandian = {
    "model": "reg_mixer_attnpl_t",
    "dataset": "ChuanDian",
    "Twindow": 600,
}

condition_rf_grid = {
    # "model": "rf",
    "Mc": 0.6,
    "Mf": 3.7,
    "Twindow": 180,
    "Tfore": 60,
    "dt": 5,
    "context_len": 1,
    # "dataset": "ChuanDian",
    # "Mag_elaps": "[5, 5.5, 6, 6.5]"
}

condition1 = {
"model": "etas",
# "bg_model": "gp_latent_bg_gpytorch",
"dataset": "CB_HAB4",
# "catalog_cfg": {
#     "mag_completeness": -0.5
#     }
}

matching_directories = find_configs_by_conditions(condition_etas_pnr_1z, checkpoints_root_dir)

In [6]:
matching_directories
# matching_directories = ["checkpoints/rf_847e64da",
#                         "checkpoints/rf_dfae9b13",
#                         "checkpoints/rf_ba2359e6",
#                         "checkpoints/rf_9ffe46be",
#                         "checkpoints/rf_af684ff4"]


['/root/autodl-tmp/em_eqf/checkpoints/etas_20251221-204926',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20251221-230212',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20251221-232500',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20251222-110009',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20251222-125538',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260101-182902',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260101-235027',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260102-115456',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260102-121710',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260102-122452',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260103-150252',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260103-150446',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260103-152502',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260103-155410',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260103-160100',
 '/root/autodl-tmp/em_eqf/checkpoints/etas_20260103-163535',
 '/root/autodl-tmp/em_eq

In [12]:
def load_metrics_from_directories(directory_list, metrics_filename):
    """Load metrics JSON files from directories if the target file exists."""
    metrics_records = []

    for directory_path in directory_list:
        metrics_file_path = os.path.join(directory_path, metrics_filename)
        if os.path.isfile(metrics_file_path):
            try:
                with open(metrics_file_path, "r", encoding="utf-8") as metrics_file:
                    metrics_data = json.load(metrics_file)

                metrics_records.append(
                    {
                        "directory_path": os.path.abspath(directory_path),
                        "metrics_data": metrics_data,
                    }
                )
            except Exception as exc:
                print(f"An error occurred while reading {metrics_file_path}: {exc}")
        else:
            print(f"File not found in directory {directory_path}: {metrics_filename}")

    return metrics_records


test_best_metrics = load_metrics_from_directories(matching_directories, "metrics_test_best_1.json")

if test_best_metrics:
    print("Found directories and their corresponding metrics content:")
    for metrics_record in test_best_metrics:
        print(f"Directory path: {metrics_record['directory_path']}")
        print(f"Metrics content: {metrics_record['metrics_data']}")
else:
    print("No directories found containing target metrics files.")


Found directories and their corresponding metrics content:
Directory path: /root/autodl-tmp/em_eqf/checkpoints/etas_20260325-103006
Metrics content: {'nll_train_time': -5.329056262969971, 'nll_train_total': -5.305012226104736, 'nll_train_bg_kl': 0.02404407039284706, 'nll_val_time': -5.207930088043213, 'nll_val_total': -5.026322364807129, 'nll_val_bg_kl': 0.18160775303840637, 'nll_test_time': -5.107245445251465, 'nll_test_total': -4.824270248413086, 'nll_test_bg_kl': 0.28297504782676697, 'num_events_train': 5495, 'num_events_val': 734, 'num_events_test': 472}
Directory path: /root/autodl-tmp/em_eqf/checkpoints/etas_20260325-110003
Metrics content: {'nll_train_time': -5.095571994781494, 'nll_train_total': -5.095571994781494, 'nll_val_time': -4.848620891571045, 'nll_val_total': -4.848620891571045, 'nll_test_time': -3.7083961963653564, 'nll_test_total': -3.7083961963653564, 'num_events_train': 5495, 'num_events_val': 734, 'num_events_test': 472}
Directory path: /root/autodl-tmp/em_eqf/chec

In [11]:
target_nll = 0.587
matching_metrics = [
    metrics_record
    for metrics_record in test_best_metrics
    if abs(float(metrics_record["metrics_data"]["nll_test_time"]) - target_nll) < 1e-3
]

print(matching_metrics)


[]


In [28]:
test_metrics = load_metrics_from_directories(matching_directories, "test_metrics.json")
val_metrics = load_metrics_from_directories(matching_directories, "val_metrics.json")


In [40]:
test_metrics


[{'folder_path': '/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_847e64da',
  'metrics_data': {'precision': 0.8,
   'recall': 0.45714285714285713,
   'f1': 0.5818181818181818,
   'auc': 0.8212389380530973,
   'fpr': 0.035398230088495575,
   'tpr': 0.45714285714285713,
   'R': 0.3657142857142857,
   'conf': 1.0,
   'threshold': 0.6364525402815664}},
 {'folder_path': '/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_dfae9b13',
  'metrics_data': {'precision': 0.5639097744360902,
   'recall': 0.9493670886075949,
   'f1': 0.7075471698113207,
   'auc': 0.6829858525688757,
   'fpr': 0.8529411764705882,
   'tpr': 0.9493670886075949,
   'R': 0.5353573807937565,
   'conf': 1.9024795333968695e-16,
   'threshold': 0.5646362918030292}},
 {'folder_path': '/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_ba2359e6',
  'metrics_data': {'precision': 0.5833333333333334,
   'recall': 0.8936170212765957,
   'f1': 0.7058823529411765,
   'auc': 0.871343085106383,
   'fpr': 0.3125,
   'tpr': 0.8936170212765957,
   '

In [ ]:
import pandas as pd

rows = []
for metrics_record in val_metrics:
    directory_path = metrics_record["directory_path"]
    metrics_data = metrics_record["metrics_data"]
    row = {"directory_path": directory_path}
    row.update(metrics_data)
    rows.append(row)

metrics_df = pd.DataFrame(rows)
# metrics_df = metrics_df.sort_values(by="directory_path").reset_index(drop=True)

metrics_df = metrics_df[["directory_path", "auc", "f1", "recall", "precision", "R"]]

print(metrics_df)


                                         folder_path       auc        f1  \
0  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.622093  0.615385   
1  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.611191  0.849206   
2  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.706827  0.764977   
3  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.879409  0.842697   
4  /home/yzzhang/pjt/chuandian_eq/checkpoints/rf_...  0.859767  0.765957   

     recall  precision         R  
0  1.000000   0.444444  0.127907  
1  1.000000   0.737931  0.025641  
2  1.000000   0.619403  0.150000  
3  0.862069   0.824176  0.586207  
4  0.870968   0.683544  0.593190  


In [53]:
metrics_df["directory_path"][3]


'/home/yzzhang/pjt/chuandian_eq/checkpoints/rf_9ffe46be'

Clean checkpoints

In [ ]:
import os
import shutil
from datetime import datetime

checkpoints_root_dir = "checkpoints"
delete_before_timestamp = "20250518-205722"
delete_before_datetime = datetime.strptime(delete_before_timestamp, "%Y%m%d-%H%M%S")

for folder_name in os.listdir(checkpoints_root_dir):
    folder_path = os.path.join(checkpoints_root_dir, folder_name)

    if os.path.isdir(folder_path) and folder_name.startswith("classifier_"):
        folder_timestamp = folder_name.split("_")[1]
        folder_datetime = datetime.strptime(folder_timestamp, "%Y%m%d-%H%M%S")

        if folder_datetime < delete_before_datetime:
            print(f"Deleting: {folder_path}")
            shutil.rmtree(folder_path)


ValueError: time data 'se' does not match format '%Y%m%d-%H%M%S'